In [1]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
grewpy.set_config('ud')
# path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/SUD_French-GSD-r2.15"
# grewpy.set_config('sud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value

# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

connected to port: 61266


In [2]:
with open("../3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

data = { k : list() for k in match_upos }
for adv, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[adv].append(formatted_features)

unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [3]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(match_upos)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(2930, 438)


In [58]:
import pyod.models.sod
import importlib
importlib.reload(pyod.models.sod)
from pyod.models.sod import SOD

model = SOD(ref_set=2, n_neighbors=5, contamination=0.1)
model.fit(X)
y_pred = model.predict(X) # binary labels (0: inliers, 1: outliers)
y_pred_proba = model.predict_proba(X) 
scores = model.decision_scores_ # raw outlier scores

In [12]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import pandas as pd

# Dimensionality reduction using t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_reduced = tsne.fit_transform(X)

# Prepare data for Plotly visualization
data = pd.DataFrame({
    "Component 1": X_reduced[:, 0],
    "Component 2": X_reduced[:, 1],
    "Word": unique_lemma,
    "Outlier": ["Outlier" if pred == 1 else "Inlier" for pred in y_pred],
    "Negative Outlier Factor": scores
})

# Create a scatter plot with Plotly
fig = go.Figure()

# Add scatter points
fig.add_trace(go.Scatter(
    x=data["Component 1"],
    y=data["Component 2"],
    mode='markers',
    marker=dict(size=10, color=["red" if o == "Outlier" else "blue" for o in data["Outlier"]]),
    text=data["Word"],  
    customdata=data["Negative Outlier Factor"],  # Inlier/Outlier status for hover
    hovertemplate="Word: %{text}<br>Score: %{customdata}<extra></extra>"
))

# Update layout
fig.update_layout(
    title="t-SNE Visualization of all lemmas with SOD - Subspace Outlier Detection",
    xaxis_title="t-SNE Component 1",
    yaxis_title="t-SNE Component 2",
    width=800,
    height=600
)

# Show the plot
fig.show()

Here I am not inverting the subspace like they say in the paper so what i do is that i am looking at features where all the points behave in the same way and my point doesn't
In this example i'm looking at faire, AUX. If i take the first feature to emerge (xparent verb form is inf) all the neighbours have a majority of no for that whereas faire has a strong majority of yes.  [grew query here](https://universal.grew.fr/?custom=67f3b500a6ea1)

To explain this we can take a simple 3D space with three axes: x,y and z. We have a set of reference points in this 3D space. These points form a cloud or a shape. For simplicity, let's say they form a flat disc lying on the x y plane. The original subspace is like the flat disc (or plane) formed by the reference points. This plane is defined by the x and y axes because the points don't vary much in the z direction (low variance in z).

In [71]:
# here because i am not ivnerting the subspace i am looking at features where the neighbours do one thing and my point does another.
# idx = adv2idx[('à', 'ADP')]
# idx = adv2idx[('cinq', 'NUM')]
idx = adv2idx[('quelque', 'DET')]

score = scores[idx]
features = np.array(model.relevant_features_[idx])
weights = np.array(model.relevant_feature_weights_[idx])
reference_points = model.ref_inds_[idx]

sort = weights.argsort()[::-1]
weights = weights[sort]
features = features[sort]
# reference_points = reference_points[sort]

print(f"Score: {score}")
print(f"Reference points: {[idx2adv[i] for i in reference_points]}")
print()

for i in range(len(features)):
    if features[i] == 1 and weights[i] > 0.0:
        print(f"{idx2feature[sort[i]]} : {weights[i]}")


Score: 0.00040752120047099255
Reference points: [('certain', 'DET'), ('tout', 'DET'), ('plusieurs', 'DET'), ('quatre', 'NUM'), ('cinq', 'NUM')]

node:X:own:Number=Sing : 2.283078428403359e-05
node:X:next:Number=Sing : 1.164835932858857e-05
node:X:parent:Number=Sing : 5.707696071008398e-06
node:X:prev:upos=PUNCT : 4.193409358291885e-06
node:X:prev:upos=ADV : 3.5236286968980416e-06
node:X:prev:Polarity=Neg : 2.9120898321471423e-06
node:X:prev:upos=SCONJ : 2.3587927640391856e-06
node:X:prev:Subject=SubjRaising : 1.8637374925741703e-06
node:X:prev:Gender=Fem : 1.4269240177520994e-06
node:X:parent:Definite=Ind : 1.0483523395729712e-06
node:X:own:rel_shallow=conj : 1.0483523395729712e-06
node:X:parent:upos=DET : 1.0483523395729712e-06
node:X:child:rel_shallow=cc : 1.0483523395729712e-06
node:X:parent:PronType=Art : 1.0483523395729712e-06
node:X:child:upos=CCONJ : 1.0483523395729712e-06
node:X:prev:Number=Plur : 1.0483523395729712e-06
node:X:parent:position=before : 1.0483523395729712e-06
nod

In [62]:
idx2adv[0]

('$', 'NOUN')

In [63]:
neighbours = [1432, 1746, 1441,  782, 2656]
for n in neighbours:
    print(f"{idx2adv[n]}")

('ha', 'NOUN')
('mm', 'NOUN')
('hectare', 'NOUN')
('cm', 'NOUN')
('tonne', 'NOUN')


Here though we're looking at the perpendicular subspace - if we had a line that sticks straight out from the flat disc perpendicular to it. So on this axis the points don't vary much but my point does. 

So here the first pattern is that the next node is a verb. If i look at faire aux you get mostly yes. with the others you get not such a clear majority as before as in you get for example 67 no and 21 yes. so all of the neighbours vary quite a lot in this feature whereas my point doesn't so much. 

In [67]:
# Assuming `idx` is the index of the point you are analyzing
features = np.array(model.relevant_features_[idx])
weights = np.array(model.relevant_feature_weights_[idx])
reference_points = model.ref_inds_[idx]

# Invert the subspace defining vector
inverted_features = 1 - features

# Sort features and weights by the weights in descending order
sort = weights.argsort()[::-1]
weights = weights[sort]
inverted_features = inverted_features[sort]

print(f"Score: {score}")
print(f"Reference points: {[idx2adv[i] for i in reference_points]}")
print()

# Print features that are relevant in the perpendicular subspace
for i in range(len(inverted_features)):
    if inverted_features[i] == 1 and weights[i] > 0.0:
        print(f"{idx2feature[sort[i]]} : {weights[i]}")


Score: 0.01000338261276736
Reference points: [('tout', 'PRON'), ('vouloir', 'VERB'), ('sembler', 'VERB'), ('laisser', 'VERB'), ('jamais', 'ADV')]

node:X:next:upos=VERB : 0.005586669617584363
node:X:next:VerbForm=Inf : 0.005485095924239071
node:X:next:Subject=Generic : 0.0016773637433167537
node:X:parent:position=before : 0.0012720299595801931
node:X:child:Number=Sing : 0.0011416556977949654
node:X:child:Person=3 : 0.0010183286934035342
node:X:child:upos=VERB : 0.0009859171335717364
node:X:child:upos=PRON : 0.0009330627031182658
node:X:child:Gender=Masc : 0.0008816643175808688
node:X:child:Emph=No : 0.0008219082342252092
node:X:child:PronType=Prs : 0.0007269741056972125
node:X:child:upos=PUNCT : 0.0007086861815513285
node:X:child:rel_shallow=punct : 0.0007086861815513285
node:X:child:rel_shallow=ccomp : 0.0006816911088073246
node:X:own:VerbForm=Inf : 0.0006552202122331072
node:X:child:VerbForm=Inf : 0.00048460086896760584
node:X:prev:Person=3 : 0.0004623233817516801
node:X:own:rel_shal

# Other visualisations of outliers

In [ ]:
outliers = []
inliers = []

for i, pred in enumerate(y_pred):
    if pred == 1:
        outliers.append(i)
    else:
        inliers.append(i)